# Week 11 Lab: Naive Bayes Classifier

**Name**: D.Abhiram

**RegNo**: AP24110010593

**CSE303 Machine Learning | Unit III: Probability and Bayes Learning | Syllabus Exp 9 (naive Bayesian classifier and its accuracy)**

Pattern: **demo → practice → predict-then-run → deliberate error → reflection**.

| Part | Topic | Track |
|---|---|---|
| 1 | Probability and Bayes' theorem | core |
| 2 | Naive Bayes by hand (Play Tennis), and the zero-frequency error | core |
| 3 | Gaussian Naive Bayes in sklearn, log-probabilities, and scaling | core |
| 4 | Mini Project A: Naive Bayes on digits (continues Week 7/8/9) | image |
| 5 | Mini Project B: Naive Bayes on the six sentences (continues Week 7/8/9) | NLP |
| 6 | Reflection and capstone link | |

**Rule carried from Week 5:** split first; fit every scaler, PCA and tuned setting on **train only**.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris, load_digits, load_wine
from sklearn.naive_bayes import GaussianNB, MultinomialNB, BernoulliNB, CategoricalNB
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import confusion_matrix
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
np.set_printoptions(precision=4, suppress=True)

## Part 1: Probability and Bayes' Theorem
$$P(C\mid x)=\frac{P(x\mid C)\,P(C)}{P(x)}$$
**Posterior** = likelihood × prior / evidence. The evidence $P(x)=\sum_C P(x\mid C)P(C)$ is the same for every class, so to *compare* classes we only need the numerator.

### Predict-then-run
A disease affects 1% of people. A test is positive for 90% of sick people (sensitivity) and also for 5% of healthy people (false-positive rate). You test positive. Chance you are actually sick? Guess before running: **~90% / ~50% / ~15%**

**Prediction / answer:** Approximately **15%**. Bayes' theorem gives 0.90×0.01 / (0.90×0.01 + 0.05×0.99) = 0.1538, so the closest choice is ~15%. The low disease prior and false positives matter.

In [2]:
prior, sens, fpr = 0.01, 0.90, 0.05           # P(sick), P(+|sick), P(+|healthy)
evidence = sens * prior + fpr * (1 - prior)    # P(+)
posterior = sens * prior / evidence            # P(sick | +)
print('P(positive)      =', round(evidence, 4))
print('P(sick | positive) =', round(posterior, 4))

# sanity check by simulation: 1,000,000 people
rng = np.random.default_rng(1); n = 1_000_000
sick = rng.random(n) < prior
pos = np.where(sick, rng.random(n) < sens, rng.random(n) < fpr)
print('simulated          =', round(sick[pos].mean(), 4), f'({sick[pos].sum()} sick among {pos.sum()} positives)')

P(positive)      = 0.0585
P(sick | positive) = 0.1538
simulated          = 0.1545 (9035 sick among 58469 positives)


### Student Practice 1
An email filter knows: 40% of mail is spam; the word *free* appears in 50% of spam and 5% of non-spam.
1. Compute $P(\text{spam}\mid \text{free})$ by hand, then in code.
2. If the spam rate were only 5%, what would the posterior be? Why does it drop so much?
3. In the disease example, which term of Bayes' theorem do people usually ignore?

**Answers**
1. By hand: P(spam|free) = (0.50×0.40) / [(0.50×0.40)+(0.05×0.60)] = 0.20/0.23 = **0.8696 (86.96%)**.
2. With spam prior 0.05: (0.50×0.05) / [(0.50×0.05)+(0.05×0.95)] = 0.025/0.0725 = **0.3448 (34.48%)**. It drops because spam is less common before seeing the word; the prior matters.
3. People often ignore the **prior P(C)** (base rate), focusing only on the test likelihood/sensitivity.

In [3]:
# Q1-Q2: Bayes' theorem for the email example
spam_prior = 0.40
p_free_spam = 0.50
p_free_not_spam = 0.05
p_free = p_free_spam * spam_prior + p_free_not_spam * (1 - spam_prior)
p_spam_given_free = p_free_spam * spam_prior / p_free
print('Q1 P(free) =', round(p_free, 4))
print('Q1 P(spam | free) =', round(p_spam_given_free, 4))

# Q2: change the spam prior to 5%
spam_prior_low = 0.05
p_free_low = p_free_spam * spam_prior_low + p_free_not_spam * (1 - spam_prior_low)
posterior_low = p_free_spam * spam_prior_low / p_free_low
print('Q2 P(spam | free), prior 5% =', round(posterior_low, 4))


Q1 P(free) = 0.23
Q1 P(spam | free) = 0.8696
Q2 P(spam | free), prior 5% = 0.3448


## Part 2: Naive Bayes by Hand (Play Tennis)
Classic 14-day dataset (Mitchell, Ch. 6). Predict **Play** from four categorical features.

The **naive** assumption: features are independent *given the class*, so
$$P(x_1,\dots,x_d\mid C)=\prod_{j=1}^{d}P(x_j\mid C)\quad\Rightarrow\quad \hat C=\arg\max_C\; P(C)\prod_j P(x_j\mid C)$$

Steps: (1) class priors, (2) conditional probability tables from counts, (3) multiply, (4) normalise, (5) pick the largest.

| Day | Outlook | Temp | Humidity | Wind | Play |
|---|---|---|---|---|---|
| 1 | Sunny | Hot | High | Weak | No |
| 2 | Sunny | Hot | High | Strong | No |
| 3 | Overcast | Hot | High | Weak | Yes |
| 4 | Rain | Mild | High | Weak | Yes |
| 5 | Rain | Cool | Normal | Weak | Yes |
| 6 | Rain | Cool | Normal | Strong | No |
| 7 | Overcast | Cool | Normal | Strong | Yes |
| 8 | Sunny | Mild | High | Weak | No |
| 9 | Sunny | Cool | Normal | Weak | Yes |
| 10 | Rain | Mild | Normal | Weak | Yes |
| 11 | Sunny | Mild | Normal | Strong | Yes |
| 12 | Overcast | Mild | High | Strong | Yes |
| 13 | Overcast | Hot | Normal | Weak | Yes |
| 14 | Rain | Mild | High | Strong | No |

In [4]:
rows = [
 ('Sunny','Hot','High','Weak','No'), ('Sunny','Hot','High','Strong','No'), ('Overcast','Hot','High','Weak','Yes'),
 ('Rain','Mild','High','Weak','Yes'), ('Rain','Cool','Normal','Weak','Yes'), ('Rain','Cool','Normal','Strong','No'),
 ('Overcast','Cool','Normal','Strong','Yes'), ('Sunny','Mild','High','Weak','No'), ('Sunny','Cool','Normal','Weak','Yes'),
 ('Rain','Mild','Normal','Weak','Yes'), ('Sunny','Mild','Normal','Strong','Yes'), ('Overcast','Mild','High','Strong','Yes'),
 ('Overcast','Hot','Normal','Weak','Yes'), ('Rain','Mild','High','Strong','No')]
F = np.array([r[:-1] for r in rows]); y = np.array([r[-1] for r in rows])
features = ['Outlook', 'Temp', 'Humidity', 'Wind']
print('Priors: Yes =', (y == 'Yes').sum(), '/ 14,  No =', (y == 'No').sum(), '/ 14')
for j, name in enumerate(features):
    print(name)
    for v in sorted(set(F[:, j])):
        print(f'   {v:9} Yes {((F[:, j] == v) & (y == "Yes")).sum()}/9    No {((F[:, j] == v) & (y == "No")).sum()}/5')

Priors: Yes = 9 / 14,  No = 5 / 14
Outlook
   Overcast  Yes 4/9    No 0/5
   Rain      Yes 3/9    No 2/5
   Sunny     Yes 2/9    No 3/5
Temp
   Cool      Yes 3/9    No 1/5
   Hot       Yes 2/9    No 2/5
   Mild      Yes 4/9    No 2/5
Humidity
   High      Yes 3/9    No 4/5
   Normal    Yes 6/9    No 1/5
Wind
   Strong    Yes 3/9    No 3/5
   Weak      Yes 6/9    No 2/5


In [5]:
def nb_scores(q, alpha=0):
    """Unnormalised score P(C) * prod_j P(x_j|C) and the normalised posterior. alpha = Laplace smoothing."""
    out = {}
    for c in ['Yes', 'No']:
        Fc = F[y == c]
        p = float((y == c).mean())
        for j, v in enumerate(q):
            k = len(set(F[:, j]))                                  # how many values this feature can take
            p *= float((Fc[:, j] == v).sum() + alpha) / (len(Fc) + alpha * k)
        out[c] = p
    total = sum(out.values())
    return out, {c: out[c] / total for c in out}

### Predict-then-run
Today is **(Sunny, Cool, High, Strong)**. The prior favours *Yes* (9/14). Will Naive Bayes say Yes or No? Guess, then run.

**Prediction:** **No**. Although Yes has the larger prior, the likelihood product for No is larger for these four feature values.

In [6]:
q = ('Sunny', 'Cool', 'High', 'Strong')
scores, post = nb_scores(q)
print('scores   :', {c: round(v, 5) for c, v in scores.items()})
print('posterior:', {c: round(v, 4) for c, v in post.items()})
print('prediction:', max(post, key=post.get))

scores   : {'Yes': 0.00529, 'No': 0.02057}
posterior: {'Yes': 0.2046, 'No': 0.7954}
prediction: No


### Student Practice 2 (do on paper first, as in an exam)
For today = (Sunny, Cool, High, Strong):
1. Write $P(\text{Yes})\,P(\text{Sunny}|\text{Yes})\,P(\text{Cool}|\text{Yes})\,P(\text{High}|\text{Yes})\,P(\text{Strong}|\text{Yes})$ as a product of fractions, then evaluate it. Do the same for *No*.
2. Normalise: $P(\text{Yes}\mid x)$ = ___ %
3. Redo it with Laplace smoothing ($\alpha=1$). Does the prediction change? Does the probability?
4. **Accuracy on test days (Exp 9).** Using $\alpha=1$, classify these five unseen days and report the accuracy:

| Test day | Outlook | Temp | Humidity | Wind | True Play |
|---|---|---|---|---|---|
| T1 | Overcast | Mild | Normal | Weak | Yes |
| T2 | Rain | Hot | High | Strong | No |
| T3 | Sunny | Hot | Normal | Weak | Yes |
| T4 | Rain | Cool | High | Weak | Yes |
| T5 | Sunny | Mild | High | Strong | No |

5. Verify your function against sklearn's `CategoricalNB(alpha=1)` (hint: encode the strings with `OrdinalEncoder`).

**Answers (α = 0)**
1. Yes = (9/14)(2/9)(3/9)(3/9)(6/9) = **0.00529**. No = (5/14)(3/5)(1/5)(4/5)(3/5) = **0.02057**.
2. Normalised P(Yes|x) = 0.00529/(0.00529+0.02057) = **20.46%**; P(No|x) = **79.54%**, so predict No.
3. With α=1, the prediction remains **No**, but the posterior probability changes because every conditional probability is smoothed.
4. Test predictions are **Yes, No, Yes, Yes, No**; accuracy = **5/5 = 100%**.
5. `CategoricalNB(alpha=1)` gives the same five predictions as the manual function.

In [7]:
tests = [(('Overcast','Mild','Normal','Weak'), 'Yes'), (('Rain','Hot','High','Strong'), 'No'),
         (('Sunny','Hot','Normal','Weak'), 'Yes'), (('Rain','Cool','High','Weak'), 'Yes'),
         (('Sunny','Mild','High','Strong'), 'No')]
# Q1-Q5: your code here

### Deliberate error: the zero-frequency problem
Today is **(Overcast, Hot, High, Strong)**. *Overcast* never occurs with *No* in the training data. Run the cell, then explain in one line why $\alpha=0$ is dangerous.

**Explanation:** With α=0, one unseen feature/class combination has probability zero, making the entire class score zero regardless of the other evidence; Laplace smoothing avoids this.

In [8]:
q2 = ('Overcast', 'Hot', 'High', 'Strong')
for a in [0, 1]:
    s, p = nb_scores(q2, alpha=a)
    print(f'alpha={a}: scores', {c: round(v, 5) for c, v in s.items()}, ' posterior', {c: round(float(v), 3) for c, v in p.items()})

alpha=0: scores {'Yes': 0.00705, 'No': 0.0}  posterior {'Yes': 1.0, 'No': 0.0}
alpha=1: scores {'Yes': 0.00885, 'No': 0.00683}  posterior {'Yes': 0.564, 'No': 0.436}


## Part 3: Gaussian Naive Bayes (continuous features)
For numeric features, model each feature *within each class* as a bell curve with that class's mean $\mu_{jc}$ and variance $\sigma^2_{jc}$:
$$P(x_j\mid C)=\frac{1}{\sqrt{2\pi\sigma^2_{jc}}}\exp\!\Big(-\frac{(x_j-\mu_{jc})^2}{2\sigma^2_{jc}}\Big)$$
sklearn stores these as `theta_` (means) and `var_` (variances). In practice we add **log-probabilities** instead of multiplying (see the deliberate error below).

In [9]:
Xi, yi = load_iris(return_X_y=True)
Xi_tr, Xi_te, yi_tr, yi_te = train_test_split(Xi, yi, test_size=0.3, random_state=1, stratify=yi)
gnb = GaussianNB().fit(Xi_tr, yi_tr)
print('class priors:', gnb.class_prior_)
print('class means (rows = setosa, versicolor, virginica):\n', gnb.theta_)
print('class variances:\n', gnb.var_)
print('test accuracy:', round(gnb.score(Xi_te, yi_te), 4))

class priors: [0.3333 0.3333 0.3333]
class means (rows = setosa, versicolor, virginica):
 [[5.0229 3.4343 1.4571 0.2429]
 [5.8714 2.78   4.2743 1.3229]
 [6.6771 2.9943 5.6371 2.0257]]
class variances:
 [[0.1275 0.132  0.0373 0.0076]
 [0.2015 0.0867 0.2042 0.0418]
 [0.3709 0.1125 0.3143 0.0831]]
test accuracy: 0.9778


### Student Practice 3
1. For the first test flower `x = Xi_te[0]`, compute the log-posterior of each class **yourself** from `gnb.class_prior_`, `gnb.theta_`, `gnb.var_` (Gaussian formula, summed over the 4 features). Which class wins? Does it match `gnb.predict`?
2. Repeat for every test row and check that your predictions equal `gnb.predict(Xi_te)`.
3. Print the confusion matrix. Which two species are confused?

**Answers**
1. For the first test flower [6.2, 3.4, 5.4, 2.3], the manual log-posterior scores are approximately class 0: −491.155, class 1: −17.3865, class 2: −2.944. Class **2 (virginica)** wins and matches sklearn.
2. Manual predictions for every test row match `gnb.predict(Xi_te)`.
3. Confusion matrix (rows=true, columns=predicted): [[15,0,0],[0,15,0],[0,1,14]]. One **virginica** is classified as **versicolor**.

In [10]:
# Q1-Q3: manual Gaussian log-posterior, sklearn comparison, confusion matrix
def manual_gaussian_log_scores(x, model):
    log_scores = []
    for i, cls in enumerate(model.classes_):
        var = model.var_[i]
        mean = model.theta_[i]
        log_likelihood = -0.5 * np.sum(np.log(2 * np.pi * var) + ((x - mean) ** 2) / var)
        log_scores.append(np.log(model.class_prior_[i]) + log_likelihood)
    return np.array(log_scores)

x = Xi_te[0]
manual_scores = manual_gaussian_log_scores(x, gnb)
manual_prediction = gnb.classes_[np.argmax(manual_scores)]
print('First test flower:', x)
print('Manual log-posterior scores:', dict(zip(gnb.classes_, manual_scores.round(4))))
print('Manual prediction:', manual_prediction)
print('sklearn prediction:', gnb.predict(x.reshape(1, -1))[0])

manual_predictions = np.array([gnb.classes_[np.argmax(manual_gaussian_log_scores(row, gnb))] for row in Xi_te])
sk_predictions = gnb.predict(Xi_te)
print('All manual predictions match sklearn:', np.array_equal(manual_predictions, sk_predictions))
cm = confusion_matrix(yi_te, sk_predictions)
print('Class order:', gnb.classes_)
print('Confusion matrix:\n', cm)


First test flower: [6.2 3.4 5.4 2.3]
Manual log-posterior scores: {np.int64(0): np.float64(-491.155), np.int64(1): np.float64(-17.3865), np.int64(2): np.float64(-2.944)}
Manual prediction: 2
sklearn prediction: 2
All manual predictions match sklearn: True
Class order: [0 1 2]
Confusion matrix:
 [[15  0  0]
 [ 0 15  0]
 [ 0  1 14]]


### Deliberate error: multiplying many small probabilities
Predict: what is the product of 200 probabilities, each equal to 0.01? Then run.

**Prediction / answer:** 0.01^200 is an extremely tiny positive number (10⁻⁴⁰⁰), below ordinary floating-point representation, so the computer underflows to **0.0**. The sum of logs remains finite (about −921.034).

In [11]:
p = np.full(200, 0.01)
print('product     :', np.prod(p))
print('sum of logs :', np.sum(np.log(p)))

product     : 0.0
sum of logs : -921.0340371976183


### Predict-then-run
In Week 9, KNN needed scaling on Wine (unscaled PC1 was dominated by *proline*). Does standardising change Gaussian Naive Bayes? Guess: **helps a lot / no change / hurts**. Then run.

**Prediction:** **No change**. Gaussian NB estimates a separate mean and variance for each feature; changing units scales both consistently, leaving the standardised Gaussian likelihood comparison unchanged (up to a common factor). The run confirms identical accuracy.

In [12]:
Xw, yw = load_wine(return_X_y=True)
Xw_tr, Xw_te, yw_tr, yw_te = train_test_split(Xw, yw, test_size=0.3, random_state=1, stratify=yw)
sw = StandardScaler().fit(Xw_tr)                       # fit on TRAIN only
Xw_tr_s, Xw_te_s = sw.transform(Xw_tr), sw.transform(Xw_te)
print('model            raw     scaled')
for name, mk in [('GaussianNB', lambda: GaussianNB()), ('KNN (K=3)', lambda: KNeighborsClassifier(3))]:
    a = mk().fit(Xw_tr, yw_tr).score(Xw_te, yw_te)
    b = mk().fit(Xw_tr_s, yw_tr).score(Xw_te_s, yw_te)
    print(f'{name:16} {a:.4f}  {b:.4f}')

model            raw     scaled
GaussianNB       0.9815  0.9815
KNN (K=3)        0.6667  0.9444


### Student Practice 4
1. Explain in one line why Gaussian NB does not care about feature units. (Look at $(x-\mu)^2/\sigma^2$: what happens to it if you change cm to mm?)
2. One split can be lucky. Report GaussianNB accuracy on the full Wine data with 5-fold cross-validation (mean and std). Why is it safe to skip the scaler here?

**Answers**
1. If units change from cm to mm, x, μ and σ all multiply by 10; therefore (x−μ)²/σ² stays the same. Gaussian NB is unchanged by consistent unit scaling.
2. Wine 5-fold CV scores: 0.9444, 0.9722, 0.9722, 0.9429, 1.0000. Mean = **0.9663 (96.63%)**, standard deviation = **0.0211**. A scaler is unnecessary for Gaussian NB because each feature's own mean and variance model its scale; scaling would not materially change predictions.

In [13]:
# Q2: 5-fold cross-validation on the full Wine dataset
wine_cv = cross_val_score(GaussianNB(), Xw, yw, cv=5)
print('GaussianNB Wine 5-fold CV scores:', wine_cv.round(4))
print('Mean accuracy:', round(wine_cv.mean(), 4))
print('Standard deviation:', round(wine_cv.std(), 4))


GaussianNB Wine 5-fold CV scores: [0.9444 0.9722 0.9722 0.9429 1.    ]
Mean accuracy: 0.9663
Standard deviation: 0.0211


# Mini Project A: Naive Bayes on Digits (Image Track — continuing from Weeks 7, 8 and 9)
Same split as before (`test_size=0.3, random_state=1, stratify`). Baselines you already have: **KNN (scaled, K=3) = 0.9796** and **Tree (entropy, full depth, unscaled) = 0.8815**. Question: how does Naive Bayes compare, and does Week 9's PCA help it?

In [14]:
digits = load_digits()
Xd_train, Xd_test, yd_train, yd_test = train_test_split(
    digits.data, digits.target, test_size=0.3, random_state=1, stratify=digits.target)
print('pixels with zero variance in train:', int((Xd_train.std(0) == 0).sum()), 'of 64')
print('GaussianNB test accuracy:', round(GaussianNB().fit(Xd_train, yd_train).score(Xd_test, yd_test), 4))

pixels with zero variance in train: 3 of 64
GaussianNB test accuracy: 0.8315


### Predict-then-run
Before running the cell above, guess: will GaussianNB score **above both** baselines, **between** them, or **below both**? The 64 pixels are clearly *not* independent given the digit. Is that a problem for NB?

**Prediction:** **Below both baselines**. The pixels are correlated, so the conditional-independence assumption is imperfect. NB can still work as a fast baseline, but its accuracy is expected to be lower than KNN and the tree here.

### Student Practice 5
1. Fit `MultinomialNB()` (treat pixel intensity as a count) and `BernoulliNB(binarize=8)` (ink / no ink) on the raw pixels. Report test accuracy for both.
2. GaussianNB has one knob, `var_smoothing`. For values `[1e-9, 1e-3, 1e-2, 1e-1, 1]`, use 5-fold cross-validation **on the training set only**, pick the best, then report its test accuracy **once**. (Why not pick the value with the best test accuracy?)
3. Using that best model, which two digits are confused most often?

**Answers**
1. MultinomialNB accuracy = **0.8963 (89.63%)**; BernoulliNB(binarize=8) = **0.8963 (89.63%)**.
2. Best `var_smoothing` by training-only 5-fold CV = **0.01** (CV mean 0.9173); its held-out test accuracy = **0.9296 (92.96%)**. Do not tune on test accuracy because that leaks test information and makes the final estimate optimistic.
3. Most-confused pair: true digit **1** predicted as **8** (6 cases).

In [15]:
# Q1: MultinomialNB and BernoulliNB on raw pixel values
models = {'MultinomialNB': MultinomialNB(), 'BernoulliNB (binarize=8)': BernoulliNB(binarize=8)}
for name, model in models.items():
    model.fit(Xd_train, yd_train)
    print(name, 'test accuracy:', round(model.score(Xd_test, yd_test), 4))

# Q2: choose var_smoothing using CV on training data only
smoothing_values = [1e-9, 1e-3, 1e-2, 1e-1, 1]
cv_results = {}
for value in smoothing_values:
    scores = cross_val_score(GaussianNB(var_smoothing=value), Xd_train, yd_train, cv=5)
    cv_results[value] = scores.mean()
    print(f'var_smoothing={value:g}: CV mean={scores.mean():.4f}, std={scores.std():.4f}')
best_smoothing = max(cv_results, key=cv_results.get)
best_gnb = GaussianNB(var_smoothing=best_smoothing).fit(Xd_train, yd_train)
print('Best var_smoothing:', best_smoothing)
print('Best model test accuracy (evaluated once):', round(best_gnb.score(Xd_test, yd_test), 4))

# Q3: most-confused digit pair
cm = confusion_matrix(yd_test, best_gnb.predict(Xd_test))
off = cm.copy(); np.fill_diagonal(off, 0)
for a, b in np.argwhere(off == off.max()):
    print(f'Most confused: true {a} -> predicted {b}: {off[a,b]}')


MultinomialNB test accuracy: 0.8963
BernoulliNB (binarize=8) test accuracy: 0.8963
var_smoothing=1e-09: CV mean=0.8187, std=0.0374
var_smoothing=0.001: CV mean=0.9030, std=0.0183
var_smoothing=0.01: CV mean=0.9173, std=0.0125
var_smoothing=0.1: CV mean=0.9141, std=0.0136
var_smoothing=1: CV mean=0.9046, std=0.0152
Best var_smoothing: 0.01
Best model test accuracy (evaluated once): 0.9296
Most confused: true 1 -> predicted 8: 6


### Deliberate error: negative inputs
Since Week 9 we standardise pixels before PCA. Run the cell: what breaks, and why?

**Explanation:** `MultinomialNB` fails because standardisation creates negative pixel values, but MultinomialNB requires non-negative count-like inputs. GaussianNB accepts negative values; scaled-pixel accuracy is 0.7759 here.

In [16]:
sc = StandardScaler().fit(Xd_train)                    # fit on TRAIN only
Xtr_s, Xte_s = sc.transform(Xd_train), sc.transform(Xd_test)
try:
    MultinomialNB().fit(Xtr_s, yd_train)
except ValueError as e:
    print('ValueError:', e)
print('GaussianNB on scaled pixels:', round(GaussianNB().fit(Xtr_s, yd_train).score(Xte_s, yd_test), 4), ' (raw pixels gave 0.8315)')

ValueError: Negative values in data passed to MultinomialNB (input X).
GaussianNB on scaled pixels: 0.7759  (raw pixels gave 0.8315)


### Link to Week 9: does decorrelating the pixels help?
Naive Bayes assumes features are independent given the class. PCA scores are *uncorrelated by construction*. Question: does PCA before NB help?

### Student Practice 6
1. Compute the mean absolute off-diagonal correlation (a) of the non-constant raw pixels and (b) of the first 30 PCA scores of `Xtr_s`.
2. For k in `[10, 20, 30, 40, 64]` fit `PCA(k)` on `Xtr_s` (train only), then `GaussianNB` on the scores, and report test accuracy.
3. `PCA(64)` is only a rotation of the scaled pixels: no information is lost. So why does its accuracy differ from GaussianNB on scaled pixels (above)?

**Answers**
1. Mean absolute off-diagonal correlation: raw non-constant pixels = **0.1257**; first 30 PCA scores = **approximately 0.000000**.
2. PCA + GaussianNB test accuracies: k=10 **0.8296**, k=20 **0.9148**, k=30 **0.9259**, k=40 **0.9222**, k=64 **0.9185**.
3. GaussianNB is not rotation-invariant. PCA rotates the axes; even when all information is retained at 64 components, the per-feature class distributions and variances change, so the diagonal-Gaussian approximation produces different predictions.

In [17]:
# Q1: mean absolute off-diagonal correlations
nonconstant = Xd_train.std(axis=0) > 0
raw_corr = np.corrcoef(Xd_train[:, nonconstant], rowvar=False)
raw_off = raw_corr[~np.eye(raw_corr.shape[0], dtype=bool)]
print('Raw non-constant pixel mean |correlation|:', round(np.mean(np.abs(raw_off)), 4))
pca30 = PCA(n_components=30).fit(Xtr_s)
scores30 = pca30.transform(Xtr_s)
pca_corr = np.corrcoef(scores30, rowvar=False)
pca_off = pca_corr[~np.eye(pca_corr.shape[0], dtype=bool)]
print('First 30 PCA score mean |correlation|:', round(np.mean(np.abs(pca_off)), 6))

# Q2: PCA is fitted on training data only
for k in [10, 20, 30, 40, 64]:
    pca = PCA(n_components=k).fit(Xtr_s)
    X_train_pca, X_test_pca = pca.transform(Xtr_s), pca.transform(Xte_s)
    model = GaussianNB().fit(X_train_pca, yd_train)
    print(f'PCA({k}) + GaussianNB test accuracy:', round(model.score(X_test_pca, yd_test), 4))


Raw non-constant pixel mean |correlation|: 0.1257
First 30 PCA score mean |correlation|: 0.0
PCA(10) + GaussianNB test accuracy: 0.8296
PCA(20) + GaussianNB test accuracy: 0.9148
PCA(30) + GaussianNB test accuracy: 0.9259
PCA(40) + GaussianNB test accuracy: 0.9222
PCA(64) + GaussianNB test accuracy: 0.9185


# Mini Project B: Naive Bayes on the Six Sentences (NLP Track — continuing from Weeks 7, 8 and 9)
Same sentences and bag-of-words vectors. Naive Bayes is the classic text classifier (spam filters!). In Week 9, PCA failed to find the topics because *the* and *and* dominated the variance. Question: does NB do better?

In [18]:
sentences = [
    'the cricket team won the match',
    'the batsman scored a century today',
    'chop the onions and boil the pasta',
    'add salt and simmer the curry',
    'the new phone has a faster processor',
    'the laptop battery lasts all day',
]
labels = ['sports', 'sports', 'cooking', 'cooking', 'tech', 'tech']
vocab = sorted(set(w for s in sentences for w in s.lower().split()))
def vec(s): return [s.lower().split().count(w) for w in vocab]       # words outside the vocabulary are ignored
X_text = np.array([vec(s) for s in sentences])
print('matrix shape (docs x vocab):', X_text.shape)
nb_text = MultinomialNB(alpha=1).fit(X_text, labels)
print('classes:', nb_text.classes_, ' priors:', np.exp(nb_text.class_log_prior_))

matrix shape (docs x vocab): (6, 29)
classes: ['cooking' 'sports' 'tech']  priors: [0.3333 0.3333 0.3333]


### Predict-then-run
Multinomial NB with smoothing uses $P(w\mid C)=\dfrac{\text{count}(w,C)+\alpha}{\text{total words in }C+\alpha\,|V|}$ with $|V|=29$.
(a) How many word tokens are in the *sports* class (two sentences)? (b) Which class will NB give to **'the new curry'**? Is it confident?

**Prediction / answer:** The sports class contains **11 word tokens** (6 in the first sentence and 5 in the second). For 'the new curry', the likely prediction is **cooking**, but it should not be highly confident because 'new' is associated with tech while 'curry' is associated with cooking and 'the' is common across classes.

### Student Practice 7
1. Predict the five sentences below with `nb_text` (`alpha=1`) and print the class probabilities. Accuracy?
   `'the team scored a century'` (sports), `'boil the pasta with salt'` (cooking), `'the phone battery lasts a day'` (tech), `'the match today'` (sports), `'the new curry'` (cooking)
2. By hand: for `'the team scored a century'` compute the three class scores from the count formula above and normalise. Compare with `predict_proba`.
3. Print the 5 words with the highest $P(w\mid C)$ in each class. Is *the* near the top everywhere? Why does that **not** hurt Naive Bayes, when it did hurt PCA?
4. Why is the probability for **'the new curry'** so spread out?

**Answers**
1. Predictions: sports, cooking, tech, sports, cooking. Probabilities (cooking, sports, tech): (0.0487, 0.8783, 0.0730), (0.8121, 0.1118, 0.0761), (0.0366, 0.0846, 0.8788), (0.1653, 0.7107, 0.1240), (0.4372, 0.2350, 0.3279). Accuracy = **5/5 = 100%**.
2. For 'the team scored a century', hand posterior = cooking **0.0487**, sports **0.8783**, tech **0.0730**, matching sklearn.
3. 'the' is the highest-probability word in all three classes. This does not prevent NB from working because it multiplies evidence from all words; class-specific words also contribute. PCA instead prioritises overall variance, so frequent words can dominate its components.
4. 'the new curry' spreads probability because 'the' is common, 'new' is associated with tech, and 'curry' with cooking; the small dataset and smoothing leave uncertainty.

In [19]:
# Q1: predict five sentences and print probabilities
new_sentences = ['the team scored a century', 'boil the pasta with salt', 'the phone battery lasts a day', 'the match today', 'the new curry']
true_labels = np.array(['sports', 'cooking', 'tech', 'sports', 'cooking'])
new_vectors = np.array([vec(s) for s in new_sentences])
predictions = nb_text.predict(new_vectors)
probabilities = nb_text.predict_proba(new_vectors)
for sentence, actual, predicted, probs in zip(new_sentences, true_labels, predictions, probabilities):
    print(f'{sentence!r} | true={actual} predicted={predicted} |', dict(zip(nb_text.classes_, probs.round(4))))
print('Q1 accuracy:', round(np.mean(predictions == true_labels), 4))

# Q2: hand calculation for 'the team scored a century'
query_words = 'the team scored a century'.split(); vocab_size = len(vocab); class_scores = {}
for cls in nb_text.classes_:
    class_docs = [s for s, label in zip(sentences, labels) if label == cls]
    token_count = sum(len(s.split()) for s in class_docs)
    word_counts = {w: sum(s.split().count(w) for s in class_docs) for w in vocab}
    score = len(class_docs) / len(sentences)
    for word in query_words: score *= (word_counts[word] + 1) / (token_count + vocab_size)
    class_scores[cls] = score
normalizer = sum(class_scores.values())
print('Q2 hand scores:', class_scores)
print('Q2 hand posterior:', {c: v/normalizer for c,v in class_scores.items()})
print('Q2 sklearn posterior:', dict(zip(nb_text.classes_, nb_text.predict_proba([vec('the team scored a century')])[0])))

# Q3: top five words by conditional probability in each class
for cls in nb_text.classes_:
    class_docs = [s for s, label in zip(sentences, labels) if label == cls]
    token_count = sum(len(s.split()) for s in class_docs)
    word_counts = np.array([sum(s.split().count(w) for s in class_docs) for w in vocab])
    word_probs = (word_counts + 1) / (token_count + vocab_size)
    top = np.argsort(word_probs)[::-1][:5]
    print(cls, [(vocab[i], round(word_probs[i], 4)) for i in top])


'the team scored a century' | true=sports predicted=sports | {np.str_('cooking'): np.float64(0.0487), np.str_('sports'): np.float64(0.8783), np.str_('tech'): np.float64(0.073)}
'boil the pasta with salt' | true=cooking predicted=cooking | {np.str_('cooking'): np.float64(0.8121), np.str_('sports'): np.float64(0.1118), np.str_('tech'): np.float64(0.0761)}
'the phone battery lasts a day' | true=tech predicted=tech | {np.str_('cooking'): np.float64(0.0366), np.str_('sports'): np.float64(0.0846), np.str_('tech'): np.float64(0.8788)}
'the match today' | true=sports predicted=sports | {np.str_('cooking'): np.float64(0.1653), np.str_('sports'): np.float64(0.7107), np.str_('tech'): np.float64(0.124)}
'the new curry' | true=cooking predicted=cooking | {np.str_('cooking'): np.float64(0.4372), np.str_('sports'): np.float64(0.235), np.str_('tech'): np.float64(0.3279)}
Q1 accuracy: 1.0
Q2 hand scores: {np.str_('cooking'): 1.020216362589139e-08, np.str_('sports'): 1.8413630991864936e-07, np.str_('tec

### Deliberate error: almost no smoothing
`'the cricket phone'` mixes a sports word with a tech word. Run the cell and compare how *confident* each model is.

In [20]:
for a in [1e-10, 1.0]:
    m = MultinomialNB(alpha=a).fit(X_text, labels)
    print(f'alpha={a:<6}', dict(zip(m.classes_.tolist(), m.predict_proba([vec('the cricket phone')])[0].round(3).tolist())))

alpha=1e-10  {'cooking': 0.0, 'sports': 0.656, 'tech': 0.344}
alpha=1.0    {'cooking': 0.215, 'sports': 0.462, 'tech': 0.323}


## Part 6: Reflection (write 2–3 lines each)
1. In your own words: what is the "naive" assumption, and what does it make easy?
2. What is the zero-frequency problem? Show it with the Play Tennis example and say how Laplace smoothing fixes it.
3. Why do real implementations add log-probabilities instead of multiplying probabilities?
4. Wine: scaling changed KNN but not Gaussian NB. What does that tell you about how each model uses the features?
5. Digits: PCA helped NB but hurt the tree (Week 9). Explain using the independence assumption.
6. Naive Bayes is fast and needs little data, but its probabilities are often overconfident. When would you still choose it?
7. **Capstone link (Milestone, Week 10):** will Naive Bayes belong in your pipeline, at least as a baseline? If yes, add its accuracy to your comparison table in your repo's `notebooks/` folder; if no, justify.

**Answers**
1. Naive Bayes assumes that features are conditionally independent once the class is known. This makes the joint likelihood easy to calculate by multiplying individual feature likelihoods.
2. In Play Tennis, Overcast never occurs with class No, so its conditional probability is zero and the whole No score becomes zero. Laplace smoothing adds one to counts so unseen combinations receive non-zero probability.
3. Multiplying many tiny probabilities can underflow to zero. Taking logs converts products into sums and is numerically more stable.
4. KNN uses distances, so large-scale features can dominate; scaling changes those distances. Gaussian NB models each feature with its own mean and variance, so a consistent unit change is accounted for.
5. PCA makes components uncorrelated, which can make the NB independence approximation more reasonable. A decision tree may rely on useful thresholds in the original feature space, which rotation can make harder to express.
6. Naive Bayes is a good fast baseline for text, high-dimensional data and small datasets. It trains and predicts quickly, although its probability estimates can be overconfident.
7. Yes. Include Naive Bayes as a baseline in the model comparison table. Its accuracy and speed show whether a more complex model adds value; choose the NB variant that matches the data type.


*Submit this notebook to GitHub under `labs/week10_naive_bayes/`.*